In [ ]:
# 1. Mount Drive and check this existing experiment (Colab GPU runtime).
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path
from google.colab import drive
import torch
import json

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/CropAdapt_Fresh")
RESULTS_DIR = PROJECT_DIR / "results"
SPLITS_DIR = PROJECT_DIR / "splits"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU, then Run all."
# This notebook resumes the existing project. It never regenerates splits.
required = [
    "resume_status.json", "splits/task_definition.json", "splits/split_protocol.json",
    "splits/source_train.csv", "splits/source_val.csv",
    "splits/target_adapt.csv", "splits/target_val.csv",
    "results/backbone_config.json", "results/lora_config.json",
    "results/loader_config.json", "results/mmd_config.json",
    "results/plantdoc_repository.json", "results/plantvillage_repository.json",
    "checkpoints/source_only/best_target_val_f1.pth",
    "checkpoints/mmd_only/latest.pth", "checkpoints/mmd_only/best_target_val_f1.pth",
]
missing = [p for p in required if not (PROJECT_DIR / p).is_file()]
assert not missing, f"Required saved project files missing: {missing}"
control_dir = CHECKPOINT_DIR / "source_continuation_control"
assert any((control_dir / p).is_file() for p in ["latest.pth", "initial.pth"]), "Existing control checkpoint is required."
print("GPU:", torch.cuda.get_device_name(0))
print("Run plan: run two new 3-epoch proposed-method experiments; resume saved progress.")
print("The original source-only run is not restarted. Target test is not evaluated.")


Mounted at /content/drive
GPU: Tesla T4
Run plan: run two new 3-epoch proposed-method experiments; resume saved progress.
The original source-only run is not restarted. Target test is not evaluated.


In [ ]:
# Restoration 1 — Restore dataset files and saved split manifests
# Preserves existing split files, results, and checkpoints.

import json
import subprocess
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/CropAdapt_Fresh")
DATA_DIR = Path("/content/cropadapt_data")
RESULTS_DIR = PROJECT_DIR / "results"
SPLITS_DIR = PROJECT_DIR / "splits"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

DATA_DIR.mkdir(parents=True, exist_ok=True)

def read_saved_json(path):
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)

resume_note = read_saved_json(PROJECT_DIR / "resume_status.json")
task_definition = read_saved_json(SPLITS_DIR / "task_definition.json")
split_protocol = read_saved_json(SPLITS_DIR / "split_protocol.json")

SEED = int(split_protocol["seed"])
class_to_idx = task_definition["class_to_idx"]
NUM_CLASSES = int(task_definition["num_classes"])
shared_classes = sorted(class_to_idx, key=class_to_idx.get)
SHARED_LABEL_MAPPING = task_definition["source_to_target"]

assert NUM_CLASSES == 16
assert sorted(class_to_idx.values()) == list(range(NUM_CLASSES))

# Load the saved splits directly; do not regenerate them.
source_train_df = pd.read_csv(SPLITS_DIR / "source_train.csv")
source_val_df = pd.read_csv(SPLITS_DIR / "source_val.csv")
target_adapt_df = pd.read_csv(SPLITS_DIR / "target_adapt.csv")
target_val_df = pd.read_csv(SPLITS_DIR / "target_val.csv")

assert {"label", "class_name", "original_class"}.isdisjoint(
    target_adapt_df.columns
), "The target adaptation manifest must remain unlabeled."

PV_ROOT = DATA_DIR / "PlantVillage-Dataset"
PD_ROOT = DATA_DIR / "PlantDoc-Dataset"
PV_COLOR_ROOT = PV_ROOT / "raw" / "color"

def git_output(root, *args):
    return subprocess.check_output(
        ["git", "-C", str(root), *args],
        text=True,
    ).strip()

def restore_repository(root, url, expected_commit, sparse_paths=None):
    if root.exists():
        if not (root / ".git").is_dir():
            raise RuntimeError(f"Existing folder is not a Git repository: {root}")

        current_commit = git_output(root, "rev-parse", "HEAD")
        if current_commit != expected_commit:
            raise RuntimeError(
                f"{root.name}: repository commit differs from the saved run.\n"
                f"Expected: {expected_commit}\n"
                f"Found:    {current_commit}"
            )

        if git_output(root, "status", "--porcelain", "--untracked-files=no"):
            raise RuntimeError(f"{root.name}: tracked files have local changes.")

        if sparse_paths:
            subprocess.run(
                ["git", "-C", str(root), "sparse-checkout",
                 "set", *sparse_paths],
                check=True,
            )
    else:
        print(f"Restoring {root.name}...", flush=True)
        subprocess.run(
            ["git", "clone", "--filter=blob:none", "--no-checkout",
             url, str(root)],
            check=True,
        )

        if sparse_paths:
            subprocess.run(
                ["git", "-C", str(root), "sparse-checkout",
                 "set", *sparse_paths],
                check=True,
            )

        # Fetch and check out the exact revision recorded for this run.
        subprocess.run(
            ["git", "-C", str(root), "fetch", "--depth", "1",
             "origin", expected_commit],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(root), "checkout", "--detach", expected_commit],
            check=True,
        )

    actual_commit = git_output(root, "rev-parse", "HEAD")
    assert actual_commit == expected_commit
    print(f"{root.name}: verified {actual_commit}")
    return actual_commit

pd_repository = read_saved_json(RESULTS_DIR / "plantdoc_repository.json")
pv_repository = read_saved_json(RESULTS_DIR / "plantvillage_repository.json")

commit = restore_repository(
    PD_ROOT,
    pd_repository["url"],
    resume_note["dataset_commits"]["plantdoc"],
)

pv_commit = restore_repository(
    PV_ROOT,
    pv_repository["url"],
    resume_note["dataset_commits"]["plantvillage"],
    sparse_paths=["raw/color", "leaf_grouping"],
)

# Check all required image paths without changing the saved manifests.
split_checks = [
    ("Source train", source_train_df, PV_ROOT, 17604),
    ("Source validation", source_val_df, PV_ROOT, 4487),
    ("Target adaptation", target_adapt_df, PD_ROOT, 1199),
    ("Target validation", target_val_df, PD_ROOT, 300),
]

for name, frame, root, expected_count in split_checks:
    assert len(frame) == expected_count, (
        f"{name}: expected {expected_count} rows, found {len(frame)}"
    )

    missing = [
        relative_path
        for relative_path in frame["relative_path"]
        if not (root / relative_path).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"{name}: {len(missing)} images missing. First examples: {missing[:3]}"
        )

    if "label" in frame.columns:
        assert frame["label"].notna().all()
        assert frame["label"].isin(range(NUM_CLASSES)).all()

    print(f"{name}: {len(frame):,} images available")

print("\nDataset restoration complete.")
print("Saved splits preserved; target test set remains unopened.")
print("Next: restore model, optimizer, AMP, and random-generator states.")


Restoring PlantDoc-Dataset...
PlantDoc-Dataset: verified 5467f6012d78d1c446145d5f582da6096f852ae8
Restoring PlantVillage-Dataset...
PlantVillage-Dataset: verified 7f7ecc7e1eaca78107e3affe7cb5abd9427e139a
Source train: 17,604 images available
Source validation: 4,487 images available
Target adaptation: 1,199 images available
Target validation: 300 images available

Dataset restoration complete.
Saved splits preserved; target test set remains unopened.
Next: restore model, optimizer, AMP, and random-generator states.


In [ ]:
# Restoration 2 — Restore the MMD model and training state

import os
import copy
import math
import random
import subprocess
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# 1. Runtime and saved configuration
# ------------------------------------------------------------
assert torch.cuda.is_available(), (
    "Select a GPU runtime in Colab before restoring this CUDA training run."
)
device = torch.device("cuda")
USE_AMP = True
AMP_INIT_SCALE = 128.0

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_generator(seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator

backbone_config = read_saved_json(RESULTS_DIR / "backbone_config.json")
lora_config = read_saved_json(RESULTS_DIR / "lora_config.json")
loader_config = read_saved_json(RESULTS_DIR / "loader_config.json")
saved_mmd_config = read_saved_json(RESULTS_DIR / "mmd_config.json")

MMD_RUN_DIR = CHECKPOINT_DIR / "mmd_only"
MMD_LATEST_PATH = MMD_RUN_DIR / "latest.pth"
MMD_BEST_PATH = MMD_RUN_DIR / "best_target_val_f1.pth"

# This is the checkpoint created by your own notebook.
restored_checkpoint = torch.load(
    MMD_LATEST_PATH,
    map_location="cpu",
    weights_only=False,
)

assert restored_checkpoint["experiment"] == "mmd_only"
assert restored_checkpoint["phase"] == "epoch_complete"
assert restored_checkpoint["class_to_idx"] == class_to_idx
assert restored_checkpoint["dino_commit"] == backbone_config["code_commit"]
assert restored_checkpoint["config"] == saved_mmd_config

MMD_CONFIG = copy.deepcopy(restored_checkpoint["config"])
mmd_history = copy.deepcopy(restored_checkpoint["history"])
mmd_best_f1 = float(restored_checkpoint["best_target_val_f1"])
completed_mmd_epoch = int(restored_checkpoint["epoch"])
next_mmd_epoch = completed_mmd_epoch + 1

assert mmd_history, "Checkpoint has no completed-epoch history."
assert [int(row["epoch"]) for row in mmd_history] == list(
    range(1, completed_mmd_epoch + 1)
)
# A completed MMD schedule is valid; checkpoint history is authoritative.
assert completed_mmd_epoch <= MMD_CONFIG["planned_epochs"]
saved_rng = restored_checkpoint["rng_state"]
assert saved_rng["cuda"] is not None
assert len(saved_rng["cuda"]) == torch.cuda.device_count(), (
    "CUDA device count differs from the saved run."
)

# ------------------------------------------------------------
# 2. Original image preprocessing and datasets
# ------------------------------------------------------------
IMAGE_SIZE = int(loader_config["image_size"])
assert IMAGE_SIZE == backbone_config["input_size"] == 224

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

to_normalized_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_geometry = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=10,
        interpolation=InterpolationMode.BILINEAR,
        fill=(124, 116, 104),
    ),
])

source_transform = transforms.Compose([
    train_geometry,
    to_normalized_tensor,
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=InterpolationMode.BICUBIC,
    ),
    to_normalized_tensor,
])

strong_appearance = transforms.Compose([
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.20,
            contrast=0.20,
            saturation=0.15,
            hue=0.02,
        )
    ], p=0.8),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
    ], p=0.2),
])

def load_rgb(path):
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")

class LabeledPlantDataset(Dataset):
    def __init__(self, manifest, root, transform):
        self.root = Path(root)
        self.paths = manifest["relative_path"].tolist()
        self.labels = manifest["label"].astype(int).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = load_rgb(self.root / self.paths[index])
        return {
            "image": self.transform(image),
            "label": self.labels[index],
        }

class UnlabeledTargetDataset(Dataset):
    def __init__(self, manifest, root):
        forbidden = {"label", "class_name", "original_class"}
        assert forbidden.isdisjoint(manifest.columns)
        self.root = Path(root)
        self.paths = manifest["relative_path"].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = load_rgb(self.root / self.paths[index])
        base = train_geometry(image)
        return {
            "weak": to_normalized_tensor(base),
            "strong": to_normalized_tensor(strong_appearance(base.copy())),
            "sample_id": index,
        }

source_train_dataset = LabeledPlantDataset(
    source_train_df, PV_ROOT, source_transform
)
source_val_dataset = LabeledPlantDataset(
    source_val_df, PV_ROOT, eval_transform
)
target_adapt_dataset = UnlabeledTargetDataset(
    target_adapt_df, PD_ROOT
)
target_val_dataset = LabeledPlantDataset(
    target_val_df, PD_ROOT, eval_transform
)

BATCH_SIZE = int(loader_config["batch_size"])
EVAL_BATCH_SIZE = int(loader_config["eval_batch_size"])
NUM_WORKERS = int(loader_config["num_workers"])
PIN_MEMORY = bool(loader_config["pin_memory"])
assert loader_config["drop_last"] is False

def make_loader(dataset, batch_size, shuffle, seed, drop_last=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=drop_last,
        worker_init_fn=seed_worker,
        generator=make_generator(seed),
        persistent_workers=False,
    )

def build_loaders(seed=SEED):
    return {
        "source_train": make_loader(
            source_train_dataset, BATCH_SIZE, True, seed
        ),
        "source_val": make_loader(
            source_val_dataset, EVAL_BATCH_SIZE, False, seed + 1
        ),
        "target_adapt": make_loader(
            target_adapt_dataset, BATCH_SIZE, True, seed + 2
        ),
        "target_val": make_loader(
            target_val_dataset, EVAL_BATCH_SIZE, False, seed + 3
        ),
    }

mmd_loaders = build_loaders(SEED)
assert set(saved_rng["loaders"]) == set(mmd_loaders)

# ------------------------------------------------------------
# 3. Restore the exact DINOv2 code revision
# ------------------------------------------------------------
DINO_REPO = Path("/content/cropadapt_models/dinov2")
DINO_REPO.parent.mkdir(parents=True, exist_ok=True)
expected_dino_commit = backbone_config["code_commit"]

if not DINO_REPO.exists():
    subprocess.run(
        ["git", "clone", "--no-checkout", "--filter=blob:none",
         backbone_config["repository"], str(DINO_REPO)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(DINO_REPO), "fetch", "--depth", "1",
         "origin", expected_dino_commit],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(DINO_REPO), "checkout",
         "--detach", expected_dino_commit],
        check=True,
    )

assert (DINO_REPO / ".git").is_dir()
dino_commit = git_output(DINO_REPO, "rev-parse", "HEAD")
assert dino_commit == expected_dino_commit, (
    "Existing DINOv2 repository does not match the checkpoint."
)
assert not git_output(
    DINO_REPO, "status", "--porcelain", "--untracked-files=no"
), "DINOv2 tracked files have local changes."

# ------------------------------------------------------------
# 4. Original model architecture
# ------------------------------------------------------------
LORA_RANK = int(lora_config["rank"])
LORA_ALPHA = int(lora_config["alpha"])
LORA_DROPOUT = float(lora_config["dropout"])
FEATURE_DIM = int(backbone_config["feature_dim"])

class LoRALinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16, dropout=0.1):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError("LoRALinear requires an nn.Linear layer.")

        self.base = base
        self.base.requires_grad_(False)
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)

        self.lora_A = nn.Parameter(
            base.weight.new_empty(rank, base.in_features)
        )
        self.lora_B = nn.Parameter(
            base.weight.new_zeros(base.out_features, rank)
        )
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        update = (
            self.dropout(x)
            @ self.lora_A.transpose(0, 1)
            @ self.lora_B.transpose(0, 1)
        )
        return self.base(x) + self.scaling * update

class CropAdaptModel(nn.Module):
    def __init__(self, pretrained_backbone):
        super().__init__()
        self.backbone = copy.deepcopy(pretrained_backbone)
        self.backbone.requires_grad_(False)

        adapted = 0
        for block in self.backbone.blocks:
            block.attn.qkv = LoRALinear(
                block.attn.qkv,
                rank=LORA_RANK,
                alpha=LORA_ALPHA,
                dropout=LORA_DROPOUT,
            )
            adapted += 1

        assert adapted == 12
        self.classifier = nn.Linear(FEATURE_DIM, NUM_CLASSES)

    def forward(self, images):
        features = self.backbone(images)
        return self.classifier(features), features

# Initialize only the architecture; all weights come from latest.pth.
set_seed(SEED)
architecture_backbone = torch.hub.load(
    str(DINO_REPO),
    backbone_config["model"],
    source="local",
    pretrained=False,
)
assert int(architecture_backbone.embed_dim) == FEATURE_DIM

mmd_model = CropAdaptModel(architecture_backbone)
mmd_model.load_state_dict(
    restored_checkpoint["model_state_dict"], strict=True
)
del architecture_backbone
mmd_model = mmd_model.to(device)

for name, parameter in mmd_model.named_parameters():
    expected_trainable = (
        name.endswith(("lora_A", "lora_B"))
        or name.startswith("classifier.")
    )
    assert parameter.requires_grad == expected_trainable, name

trainable_count = sum(
    p.numel() for p in mmd_model.parameters() if p.requires_grad
)
assert trainable_count == lora_config["trainable_parameters"] == 153616
assert sum(p.numel() for p in mmd_model.parameters()) == (
    lora_config["total_parameters"]
)

# ------------------------------------------------------------
# 5. Optimizer and AMP restoration
# ------------------------------------------------------------
mmd_optimizer = torch.optim.AdamW(
    [p for p in mmd_model.parameters() if p.requires_grad],
    lr=MMD_CONFIG["lr"],
    weight_decay=MMD_CONFIG["weight_decay"],
)
mmd_optimizer.load_state_dict(
    restored_checkpoint["optimizer_state_dict"]
)

mmd_scaler = torch.amp.GradScaler(
    "cuda", enabled=USE_AMP, init_scale=AMP_INIT_SCALE
)
mmd_scaler.load_state_dict(restored_checkpoint["scaler_state_dict"])

mmd_optimizer.zero_grad(set_to_none=True)
mmd_model.eval()

# ------------------------------------------------------------
# 6. Restore randomness LAST, after model construction
# ------------------------------------------------------------
random.setstate(saved_rng["python"])
np.random.set_state(saved_rng["numpy"])
torch.set_rng_state(saved_rng["torch"])
torch.cuda.set_rng_state_all(saved_rng["cuda"])

for name, loader in mmd_loaders.items():
    loader.generator.set_state(saved_rng["loaders"][name])

assert torch.equal(torch.get_rng_state(), saved_rng["torch"])
for actual, expected in zip(
    torch.cuda.get_rng_state_all(), saved_rng["cuda"]
):
    assert torch.equal(actual, expected)
for name, loader in mmd_loaders.items():
    assert torch.equal(
        loader.generator.get_state(), saved_rng["loaders"][name]
    )

del saved_rng, restored_checkpoint

print("GPU:", torch.cuda.get_device_name(0))
print("DINOv2 commit:", dino_commit)
print("Model weights: restored with strict key matching")
print("Trainable parameters:", f"{trainable_count:,}")
print("Optimizer: restored")
print("AMP scale:", mmd_scaler.get_scale())
print("Python, NumPy, PyTorch, CUDA and loader RNG states: restored")
print("Completed MMD epoch:", completed_mmd_epoch)
print("MMD epochs remaining:", max(0, MMD_CONFIG["planned_epochs"] - completed_mmd_epoch))
print("Planned total epochs:", MMD_CONFIG["planned_epochs"])
print(f"Best MMD target-validation macro-F1: {mmd_best_f1:.6f}")
print("\nRestoration complete. Training has not started.")


/content/cropadapt_models/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/cropadapt_models/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/cropadapt_models/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


GPU: Tesla T4
DINOv2 commit: 7764ea0f912e53c92e82eb78a2a1631e92725fc8
Model weights: restored with strict key matching
Trainable parameters: 153,616
Optimizer: restored
AMP scale: 256.0
Python, NumPy, PyTorch, CUDA and loader RNG states: restored
Completed MMD epoch: 3
MMD epochs remaining: 0
Planned total epochs: 3
Best MMD target-validation macro-F1: 0.376320

Restoration complete. Training has not started.


In [ ]:
# 4. Original evaluation and baseline MMD definitions; no old training is executed.
import time

import random

import copy

import numpy as np

import pandas as pd

import torch

import torch.nn as nn

import torch.nn.functional as F

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

class MultiKernelMMD(nn.Module):
    def __init__(self, kernel_scales=(0.25, 0.5, 1.0, 2.0, 4.0)):
        super().__init__()
        self.register_buffer(
            "kernel_scales",
            torch.tensor(kernel_scales, dtype=torch.float32),
        )

    def forward(self, source_features, target_features):
        if source_features.ndim != 2 or target_features.ndim != 2:
            raise ValueError("Expected [batch, features] matrices.")
        if source_features.shape[1] != target_features.shape[1]:
            raise ValueError("Source and target feature dimensions differ.")
        if min(len(source_features), len(target_features)) == 0:
            raise ValueError("MMD requires nonempty batches.")

        with torch.autocast(
            device_type=source_features.device.type,
            enabled=False,
        ):
            source = F.normalize(source_features.float(), dim=1)
            target = F.normalize(target_features.float(), dim=1)
            combined = torch.cat([source, target], dim=0)

            distances = (
                combined[:, None, :] - combined[None, :, :]
            ).square().sum(dim=-1)

            with torch.no_grad():
                upper = torch.triu_indices(
                    len(combined), len(combined),
                    offset=1, device=combined.device,
                )
                pair_distances = distances[upper[0], upper[1]]
                positive = pair_distances[pair_distances > 1e-8]
                bandwidth = (
                    positive.median()
                    if positive.numel()
                    else distances.new_tensor(1.0)
                ).clamp_min(1e-6)

            kernels = torch.exp(
                -distances.unsqueeze(0)
                / (
                    bandwidth * self.kernel_scales.to(combined.device)
                )[:, None, None]
            ).mean(dim=0)

            n_source = len(source)
            return (
                kernels[:n_source, :n_source].mean()
                + kernels[n_source:, n_source:].mean()
                - 2 * kernels[:n_source, n_source:].mean()
            )

@torch.no_grad()
def evaluate_model(model, loader, description="Validation", max_batches=None):
    was_training = model.training
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_labels = []
    all_predictions = []

    try:
        progress = tqdm(
            loader,
            desc=description,
            total=(
                len(loader) if max_batches is None
                else min(max_batches, len(loader))
            ),
            leave=False,
        )

        for batch_index, batch in enumerate(progress):
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits, _ = model(images)

            logits = logits.float()
            if not torch.isfinite(logits).all().item():
                raise RuntimeError("Non-finite evaluation logits.")

            loss_sum = F.cross_entropy(logits, labels, reduction="sum")
            predictions = logits.argmax(dim=1)

            total_loss += loss_sum.item()
            total_samples += labels.numel()
            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())

            if max_batches is not None and batch_index + 1 >= max_batches:
                break

        if total_samples == 0:
            raise RuntimeError("Evaluation loader produced no samples.")

        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels,
            all_predictions,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0,
        )

        metrics = {
            "loss": total_loss / total_samples,
            "accuracy": float(accuracy_score(all_labels, all_predictions)),
            "macro_f1": float(f1),
            "macro_precision": float(precision),
            "macro_recall": float(recall),
            "num_samples": total_samples,
        }

        details = {
            "labels": np.asarray(all_labels, dtype=np.int64),
            "predictions": np.asarray(all_predictions, dtype=np.int64),
            "confusion_matrix": confusion_matrix(
                all_labels, all_predictions,
                labels=list(range(NUM_CLASSES)),
            ),
        }
        return metrics, details

    finally:
        model.train(was_training)

In [ ]:
# 5. Proposed method configuration and matched source-only initialization.
# Research prototype: CE + adaptive feature-weighted MMD + optional source margin.
# Target validation is used ONLY for reporting/checkpoint selection.
import hashlib
import gc
import time

DG_CONFIG = {
    'version': 'dgwmmd_v1', 'epochs': 3, 'seed': SEED,
    'lr': float(MMD_CONFIG['lr']), 'weight_decay': float(MMD_CONFIG['weight_decay']),
    'lambda_max': 0.1, 'warmup_epochs': float(MMD_CONFIG['warmup_epochs']),
    'gamma': 1.0, 'weight_floor': 0.2, 'moment_decay': 0.95,
    'prototype_decay': 0.95, 'margin': 0.15, 'eps': 1e-6,
    'kernel_scales': [0.25, 0.5, 1.0, 2.0, 4.0],
    'separation_refresh': 'full_source_train_eval_at_epoch_end',
    'statistics_transform': 'original_eval_transform',
}
DG_RUNS = [('dgwmmd_v1', 0.0), ('dgwmmd_margin_v1', 0.1)]
DG_RESULTS = RESULTS_DIR / 'proposed_dgwmmd_v1'
DG_RESULTS.mkdir(parents=True, exist_ok=True)

def dg_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def dg_save(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    torch.save(obj, tmp); tmp.replace(path)

def dg_csv(frame, path):
    path = Path(path); tmp = path.with_suffix('.tmp')
    frame.to_csv(tmp, index=False); tmp.replace(path)

DG_SOURCE_PATH = CHECKPOINT_DIR / 'source_only' / 'best_target_val_f1.pth'
DG_SOURCE = torch.load(DG_SOURCE_PATH, map_location='cpu', weights_only=False)
assert DG_SOURCE['class_to_idx'] == class_to_idx
# Match the random streams of the previous MMD 0.1 initialization when available.
DG_INIT_PATH = CHECKPOINT_DIR / 'mmd_weight_0p1' / 'initial.pth'
if not DG_INIT_PATH.exists():
    DG_INIT_PATH = CHECKPOINT_DIR / 'source_continuation_control' / 'initial.pth'
DG_INIT = torch.load(DG_INIT_PATH, map_location='cpu', weights_only=False)
assert DG_INIT['epoch'] == 0 and DG_INIT['class_to_idx'] == class_to_idx
assert DG_INIT['dino_commit'] == dino_commit
assert all(torch.equal(v, DG_INIT['model_state_dict'][k])
           for k,v in DG_SOURCE['model_state_dict'].items()), 'Initialization differs from source-only best.'
assert DG_INIT['config']['lr'] == DG_CONFIG['lr']
assert DG_INIT['config']['weight_decay'] == DG_CONFIG['weight_decay']
assert DG_INIT['config']['warmup_epochs'] == DG_CONFIG['warmup_epochs']
assert not DG_INIT['optimizer_state_dict']['state'], 'Baseline optimizer was not fresh; inspect before comparing.'
DG_PROVENANCE = {
    'source_checkpoint_sha256': dg_hash(DG_SOURCE_PATH),
    'initial_checkpoint_sha256': dg_hash(DG_INIT_PATH),
    'source_train_sha256': dg_hash(SPLITS_DIR / 'source_train.csv'),
    'target_adapt_sha256': dg_hash(SPLITS_DIR / 'target_adapt.csv'),
    'class_to_idx': class_to_idx, 'dino_commit': dino_commit,
}
# Reuse the architecture, keeping only one active training model on GPU.
mmd_optimizer.zero_grad(set_to_none=True)
mmd_model.cpu()
del mmd_optimizer, mmd_scaler
DG_TEMPLATE = mmd_model
DG_TEMPLATE.load_state_dict(DG_SOURCE['model_state_dict'], strict=True)
torch.cuda.empty_cache()
print('Starting source-only epoch:', DG_SOURCE['epoch'])
print('Two separate 3-epoch runs. No old experiments will train.')
print('Mid-epoch interruption: restart that epoch. Completed training resumes at validation.')



Starting source-only epoch: 3
Two separate 3-epoch runs. No old experiments will train.
Mid-epoch interruption: restart that epoch. Completed training resumes at validation.


In [ ]:
# 6. Equations and deterministic source-only reference statistics.
def dg_mmd(zs, zt, weights):
    # Inputs are already L2-normalized; do NOT normalize after weighting.
    with torch.autocast(device_type=zs.device.type, enabled=False):
        z = torch.cat([zs.float(), zt.float()]) * weights.detach().sqrt()
        dist = (z[:,None,:] - z[None,:,:]).square().sum(-1)
        with torch.no_grad():
            ix = torch.triu_indices(len(z),len(z),1,device=z.device)
            pos = dist[ix[0],ix[1]]; pos = pos[pos > 1e-8]
            bw = (pos.median() if pos.numel() else dist.new_tensor(1.)).clamp_min(1e-6)
        scales = z.new_tensor(DG_CONFIG['kernel_scales'])
        k = torch.exp(-dist[None] / (bw*scales[:,None,None])).mean(0)
        ns = len(zs)
        return k[:ns,:ns].mean()+k[ns:,ns:].mean()-2*k[:ns,ns:].mean()

def dg_weights(sm, sv, tm, tv, q):
    eps = DG_CONFIG['eps']
    d = (sm-tm).square() / (sv+tv+eps)
    r = d / (1+DG_CONFIG['gamma']*q)
    if r.sum() <= eps: return torch.ones_like(r)
    rho = DG_CONFIG['weight_floor']
    return (rho+(1-rho)*r.numel()*r/r.sum()).detach()

def dg_margin(z, labels, prototypes):
    distances = 1-z @ F.normalize(prototypes.detach(),dim=1).T
    own = distances.gather(1,labels[:,None]).squeeze(1)
    other = distances.masked_fill(F.one_hot(labels,NUM_CLASSES).bool(),float('inf')).min(1).values
    losses = F.relu(DG_CONFIG['margin']+own-other)
    return torch.stack([losses[labels==c].mean() for c in labels.unique()]).mean()

@torch.no_grad()
def dg_source_statistics(model):
    # Local generator prevents this measurement from consuming training loader RNG.
    loader = make_loader(LabeledPlantDataset(source_train_df,PV_ROOT,eval_transform),
                         EVAL_BATCH_SIZE,False,SEED+710)
    counts = torch.zeros(NUM_CLASSES,device=device,dtype=torch.float64)
    sums = torch.zeros(NUM_CLASSES,FEATURE_DIM,device=device,dtype=torch.float64)
    squares = torch.zeros_like(sums)
    was_training = model.training; model.eval()
    try:
        for batch in tqdm(loader,desc='Source training feature statistics',leave=False):
            y = batch['label'].to(device)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP):
                _, f = model(batch['image'].to(device,non_blocking=True))
            z = F.normalize(f.float(),dim=1).double()
            counts.index_add_(0,y,torch.ones_like(y,dtype=torch.float64))
            sums.index_add_(0,y,z); squares.index_add_(0,y,z.square())
    finally: model.train(was_training)
    assert (counts>0).all(), 'Missing source class.'
    means = sums/counts[:,None]
    variances = (squares/counts[:,None]-means.square()).clamp_min(0)
    between = means.var(0,unbiased=False); within = variances.mean(0)
    eps = DG_CONFIG['eps']
    return {'q': (between/(within+eps)).float().cpu(),
            'prototypes': F.normalize(means.float(),dim=1).cpu(),
            'J': float(between.sum()/(within.sum()+eps)),
            'mean': (sums.sum(0)/counts.sum()).float().cpu(),
            'var': (squares.sum(0)/counts.sum()-(sums.sum(0)/counts.sum()).square()).clamp_min(0).float().cpu()}

DG_REF_PATH = DG_RESULTS / 'source_reference.pth'
if DG_REF_PATH.exists():
    reference = torch.load(DG_REF_PATH,map_location='cpu',weights_only=False)
    assert reference['provenance'] == DG_PROVENANCE
    assert reference['config'] == DG_CONFIG
else:
    DG_TEMPLATE.to(device)
    reference = {'provenance':DG_PROVENANCE,'config':DG_CONFIG,
                 'statistics':dg_source_statistics(DG_TEMPLATE)}
    dg_save(reference,DG_REF_PATH); DG_TEMPLATE.cpu(); torch.cuda.empty_cache()
DG_REF = reference['statistics']
assert DG_REF['J'] > 0 and torch.isfinite(DG_REF['q']).all()
print('Reference class-separation score:',DG_REF['J'])
print('Reference statistics use only source_train images and labels.')



Source training feature statistics:   0%|          | 0/551 [00:00<?, ?it/s]

Reference class-separation score: 1.661203956927867
Reference statistics use only source_train images and labels.


In [ ]:
# 7. Resumable training engine; controllers and all RNG streams are checkpointed.
def dg_rng(loaders):
    return {'python':random.getstate(),'numpy':np.random.get_state(),
            'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all(),
            'loaders':{k:v.generator.get_state() for k,v in loaders.items()}}

def dg_restore_rng(rng,loaders):
    random.setstate(rng['python']); np.random.set_state(rng['numpy'])
    torch.set_rng_state(rng['torch']); torch.cuda.set_rng_state_all(rng['cuda'])
    for k,v in loaders.items(): v.generator.set_state(rng['loaders'][k])

def dg_train_epoch(model,opt,scaler,loaders,ctrl,epoch,eta):
    model.train(); target_it = iter(loaders['target_adapt'])
    pars = [p for p in model.parameters() if p.requires_grad]
    q = DG_REF['q'].to(device); proto = ctrl['prototypes'].to(device)
    moment = {k:v.to(device) for k,v in ctrl['moments'].items()}
    factor = min(1.,max(0.,ctrl['J']/DG_REF['J']))
    sums = dict(loss=0.,ce=0.,mmd=0.,margin=0.,weight=0.)
    count=updates=skipped=consecutive=0; start=time.perf_counter()
    warm = max(1.,DG_CONFIG['warmup_epochs']*len(loaders['source_train']))
    try:
        for step,b in enumerate(tqdm(loaders['source_train'],desc=f'Proposed | epoch {epoch} | margin={eta}')):
            try: t=next(target_it)
            except StopIteration:
                target_it=iter(loaders['target_adapt']); t=next(target_it)
            y=b['label'].to(device); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP):
                logits,sf=model(b['image'].to(device,non_blocking=True))
                _,tf=model(t['weak'].to(device,non_blocking=True))
            zs=F.normalize(sf.float(),dim=1); zt=F.normalize(tf.float(),dim=1)
            ce=F.cross_entropy(logits.float(),y)
            # EMA of first and second raw moments; detach controller from autograd.
            with torch.no_grad():
                for prefix,z in [('s',zs),('t',zt)]:
                    for suffix,v in [('mean',z.mean(0)),('second',z.square().mean(0))]:
                        key=prefix+'_'+suffix
                        moment[key]=v.detach() if key not in moment else DG_CONFIG['moment_decay']*moment[key]+(1-DG_CONFIG['moment_decay'])*v.detach()
                sv=(moment['s_second']-moment['s_mean'].square()).clamp_min(0)
                tv=(moment['t_second']-moment['t_mean'].square()).clamp_min(0)
                w=dg_weights(moment['s_mean'],sv,moment['t_mean'],tv,q)
            alignment=dg_mmd(zs,zt,w)
            margin=dg_margin(zs,y,proto) if eta else zs.new_zeros(())
            weight=DG_CONFIG['lambda_max']*factor*min(1.,((epoch-1)*len(loaders['source_train'])+step+1)/warm)
            loss=ce+weight*alignment+eta*margin
            if not torch.isfinite(loss): raise RuntimeError('Non-finite loss; rerun to restore epoch boundary.')
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            if any(p.grad is None for p in pars): raise RuntimeError('Missing trainable gradient.')
            finite=all(torch.isfinite(p.grad).all().item() for p in pars)
            if finite:
                torch.nn.utils.clip_grad_norm_(pars,1.,error_if_nonfinite=True)
            elif not scaler.is_enabled(): raise RuntimeError('Non-finite gradients without AMP.')
            scaler.step(opt); scaler.update()
            if finite:
                updates+=1; consecutive=0
                with torch.no_grad():
                    for c in y.unique():
                        center=F.normalize(zs[y==c].mean(0),dim=0)
                        proto[c]=F.normalize(DG_CONFIG['prototype_decay']*proto[c]+(1-DG_CONFIG['prototype_decay'])*center,dim=0)
            else:
                skipped+=1; consecutive+=1
                if consecutive>=8: raise RuntimeError('Eight consecutive skipped updates.')
            size=len(y); count+=size
            for k,v in [('loss',loss.item()),('ce',ce.item()),('mmd',alignment.item()),('margin',margin.item()),('weight',weight)]: sums[k]+=size*v
    finally: del target_it
    assert updates>0
    ctrl['prototypes']=proto.detach().cpu()
    ctrl['moments']={k:v.detach().cpu() for k,v in moment.items()}
    return {**{'train_'+k:v/count for k,v in sums.items()},'successful_updates':updates,
            'skipped_updates':skipped,'separation_factor_used':factor,'train_seconds':time.perf_counter()-start}

def dg_run(name,eta):
    run=CHECKPOINT_DIR/name; run.mkdir(parents=True,exist_ok=True)
    latest=run/'latest.pth'; pending=run/'pending_validation.pth'; best_path=run/'best_target_val_f1.pth'
    cfg={**DG_CONFIG,'eta':eta,'experiment':name}
    model=copy.deepcopy(DG_TEMPLATE).to(device)
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=cfg['lr'],weight_decay=cfg['weight_decay'])
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP,init_scale=AMP_INIT_SCALE)
    loaders=build_loaders(SEED)
    def check(saved):
        assert saved['config']==cfg and saved['provenance']==DG_PROVENANCE, 'Saved run differs; use a new experiment name.'
    def pack(epoch,phase,metrics=None):
        return {'experiment':name,'epoch':epoch,'phase':phase,'config':cfg,'provenance':DG_PROVENANCE,
                'class_to_idx':class_to_idx,'dino_commit':dino_commit,
                'model_state_dict':model.state_dict(),'optimizer_state_dict':opt.state_dict(),
                'scaler_state_dict':scaler.state_dict(),'rng_state':dg_rng(loaders),
                'controller':copy.deepcopy(ctrl),'history':copy.deepcopy(history),
                'best_target_val_f1':best,'train_metrics':metrics}
    try:
        if latest.exists(): saved=torch.load(latest,map_location='cpu',weights_only=False); check(saved)
        else:
            model.load_state_dict(DG_SOURCE['model_state_dict'],strict=True)
            # Fresh optimizer, matched AMP and random state from original adaptation initialization.
            scaler.load_state_dict(DG_INIT['scaler_state_dict'])
            dg_restore_rng(DG_INIT['rng_state'],loaders)
            ctrl={'J':DG_REF['J'],'prototypes':DG_REF['prototypes'].clone(),
                  'moments':{'s_mean':DG_REF['mean'].clone(),'s_second':DG_REF['var']+DG_REF['mean'].square()}}
            history=[]; best=-1.
            saved=pack(0,'epoch_complete'); dg_save(saved,latest)
        if pending.exists():
            p=torch.load(pending,map_location='cpu',weights_only=False); check(p)
            if p['epoch']>saved['epoch']:
                assert p['epoch']==saved['epoch']+1 and p['phase']=='trained_pending_validation'
                saved=p
        model.load_state_dict(saved['model_state_dict'],strict=True)
        opt.load_state_dict(saved['optimizer_state_dict']); scaler.load_state_dict(saved['scaler_state_dict'])
        ctrl=copy.deepcopy(saved['controller']); history=copy.deepcopy(saved['history']); best=float(saved['best_target_val_f1'])
        dg_restore_rng(saved['rng_state'],loaders)
        pending_epoch=saved['epoch'] if saved['phase']=='trained_pending_validation' else None
        start_epoch=saved['epoch'] if pending_epoch else saved['epoch']+1
        pending_metrics=saved.get('train_metrics'); del saved
        for epoch in range(start_epoch,cfg['epochs']+1):
            if epoch==pending_epoch:
                print(name,epoch,': saved training found; resuming measurements.')
                tm=pending_metrics
            else:
                tm=dg_train_epoch(model,opt,scaler,loaders,ctrl,epoch,eta)
                dg_save(pack(epoch,'trained_pending_validation',tm),pending)
            sm,_=evaluate_model(model,loaders['source_val'],name+' source validation')
            vm,_=evaluate_model(model,loaders['target_val'],name+' target validation')
            # Comparable full-source statistics, no noisy per-batch class scatter.
            # J from the end of this epoch controls the NEXT epoch's alignment strength.
            stats=dg_source_statistics(model); ctrl['J']=stats['J']
            row={'epoch':epoch,**tm,'source_separation_J':ctrl['J'],
                 **{'source_val_'+k:v for k,v in sm.items()},**{'target_val_'+k:v for k,v in vm.items()}}
            history.append(row)
            improved=vm['macro_f1']>best
            best=max(best,vm['macro_f1'])
            completed=pack(epoch,'epoch_complete')
            if improved: dg_save(completed,best_path)
            dg_save(completed,latest)
            dg_csv(pd.DataFrame(history),DG_RESULTS/(name+'_history.csv'))
            print(f"{name} epoch {epoch}: accuracy={vm['accuracy']:.4f}, macro-F1={vm['macro_f1']:.4f}, best={improved}")
        dg_csv(pd.DataFrame(history),DG_RESULTS/(name+'_history.csv'))
        print(name,'complete. Best validation macro-F1:',best)
    finally:
        model.cpu(); del model,opt,scaler,loaders; gc.collect(); torch.cuda.empty_cache()



In [ ]:
# 8. Small numerical/gradient checks before spending GPU training time.
# Synthetic data only; does not run optimizer steps or change model weights.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(741)
    a=torch.randn(5,FEATURE_DIM,requires_grad=True); b=torch.randn(7,FEATURE_DIM,requires_grad=True)
    za=F.normalize(a,dim=1); zb=F.normalize(b,dim=1)
    one=torch.ones(FEATURE_DIM)
    assert torch.allclose(dg_mmd(za,zb,one),MultiKernelMMD()(a,b),atol=2e-6)
    weights=dg_weights(torch.zeros(FEATURE_DIM),one,torch.ones(FEATURE_DIM),one,one)
    assert torch.allclose(weights.mean(),torch.tensor(1.)) and weights.min()>=DG_CONFIG['weight_floor']
    prot=F.normalize(torch.randn(NUM_CLASSES,FEATURE_DIM),dim=1)
    loss=dg_mmd(za,zb,weights)+0.1*dg_margin(za,torch.arange(5),prot)
    loss.backward()
    assert torch.isfinite(a.grad).all() and torch.isfinite(b.grad).all()
print('Weighted MMD equivalence, weight constraints, and gradient checks passed.')



Weighted MMD equivalence, weight constraints, and gradient checks passed.


In [ ]:
# 9. Run the two independent experiments. Completed epochs are automatically skipped.
for dg_name,dg_eta in DG_RUNS:
    dg_run(dg_name,dg_eta)
print('Both experiments complete. Target test remains unopened.')



Proposed | epoch 1 | margin=0.0:   0%|          | 0/1101 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


dgwmmd_v1 source validation:   0%|          | 0/141 [00:00<?, ?it/s]

dgwmmd_v1 target validation:   0%|          | 0/10 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78770c542160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x78770c542160>
if w.is_alive():Traceback (most recent call last):
    self._shutdown_workers()Exception ignored in: 
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78770c542160>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_wor

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78770c542160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78770c542160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

dgwmmd_margin_v1 target validation:   0%|          | 0/10 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78770c542160>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x78770c542160>    
Traceback (most recent call last):
self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
if w.is_alive():    
self._shutdown_workers()  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive

    assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

AssertionError    : can only test a child process
if w.is_alive():
  File "/usr/lib/

Source training feature statistics:   0%|          | 0/551 [00:00<?, ?it/s]

dgwmmd_margin_v1 epoch 3: accuracy=0.4833, macro-F1=0.4168, best=True
dgwmmd_margin_v1 complete. Best validation macro-F1: 0.41680563221551176
Both experiments complete. Target test remains unopened.


In [ ]:
# 10. Compare with saved baselines using the same THREE adaptation-epoch budget.
# Histories are checkpoint-authoritative. Later baseline epochs are excluded here.
rows=[]
for name in ['source_only','mmd_only','source_continuation_control','mmd_weight_0p1']+[x[0] for x in DG_RUNS]:
    path=CHECKPOINT_DIR/name/'latest.pth'
    if not path.exists():
        print('Comparison skipped; checkpoint missing:',name); continue
    saved=torch.load(path,map_location='cpu',weights_only=False)
    hist=saved.get('history',[])
    eligible=[r for r in hist if name=='source_only' or int(r['epoch'])<=DG_CONFIG['epochs']]
    if not eligible:
        print('Comparison skipped; no eligible history:',name); continue
    selected=max(eligible,key=lambda r:float(r['target_val_macro_f1']))
    rows.append({'experiment':name,'selected_epoch':int(selected['epoch']),
                 'accuracy':selected['target_val_accuracy'],'macro_f1':selected['target_val_macro_f1'],
                 'comparison':'source pretraining' if name=='source_only' else 'best within adaptation epochs 1-3'})
comparison=pd.DataFrame(rows).sort_values('macro_f1',ascending=False,kind='stable')
dg_csv(comparison,DG_RESULTS/'matched_budget_validation_comparison.csv')
print('\nMATCHED-BUDGET VALIDATION COMPARISON'); print(comparison.to_string(index=False))
# Fresh per-class evaluation for proposed models and original baseline.
report_model=copy.deepcopy(DG_TEMPLATE).to(device)
try:
    for name in ['source_only']+[x[0] for x in DG_RUNS]:
        path=CHECKPOINT_DIR/name/'best_target_val_f1.pth'
        saved=torch.load(path,map_location='cpu',weights_only=False)
        report_model.load_state_dict(saved['model_state_dict'],strict=True)
        metrics,details=evaluate_model(report_model,make_loader(target_val_dataset,EVAL_BATCH_SIZE,False,SEED+811),name)
        pr,re,f1,support=precision_recall_fscore_support(details['labels'],details['predictions'],labels=list(range(NUM_CLASSES)),zero_division=0)
        dg_csv(pd.DataFrame({'class_name':shared_classes,'support':support,'precision':pr,'recall':re,'f1':f1}),DG_RESULTS/(name+'_per_class.csv'))
        dg_csv(pd.DataFrame({'relative_path':target_val_df['relative_path'],'label':details['labels'],'prediction':details['predictions']}),DG_RESULTS/(name+'_predictions.csv'))
        print(name,'zero-recall classes:',int((re==0).sum()))
finally:
    report_model.cpu(); del report_model; torch.cuda.empty_cache()
(DG_RESULTS/'method_config.json').write_text(json.dumps({'config':DG_CONFIG,'runs':DG_RUNS,'provenance':DG_PROVENANCE,
    'selection_split':'target_val','target_test_evaluated':False,'novelty_status':'proposed course-project modification; originality not established'},indent=2))
print('Reports:',DG_RESULTS)
print('Target test untouched. Validation gains are exploratory, not final test results.')



MATCHED-BUDGET VALIDATION COMPARISON
                 experiment  selected_epoch  accuracy  macro_f1                        comparison
                  dgwmmd_v1               3  0.490000  0.424752 best within adaptation epochs 1-3
           dgwmmd_margin_v1               3  0.483333  0.416806 best within adaptation epochs 1-3
             mmd_weight_0p1               3  0.476667  0.410413 best within adaptation epochs 1-3
                source_only               3  0.450000  0.396580                source pretraining
source_continuation_control               3  0.436667  0.379944 best within adaptation epochs 1-3
                   mmd_only               1  0.450000  0.376320 best within adaptation epochs 1-3


source_only:   0%|          | 0/10 [00:00<?, ?it/s]

source_only zero-recall classes: 2


dgwmmd_v1:   0%|          | 0/10 [00:00<?, ?it/s]

dgwmmd_v1 zero-recall classes: 3


dgwmmd_margin_v1:   0%|          | 0/10 [00:00<?, ?it/s]

dgwmmd_margin_v1 zero-recall classes: 3
Reports: /content/drive/MyDrive/CropAdapt_Fresh/results/proposed_dgwmmd_v1
Target test untouched. Validation gains are exploratory, not final test results.
